# Feature Track 3: Maintain a Conversation

# Setup

In [12]:
from loguru import logger
import sys

from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.agents.base import QueryWithContext
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_vector_store,
    build_llm,
    build_agent,
    SYSTEM_PROMPT,
    EMBEDDING_MODEL,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")  # hides DEBUG

4

In [13]:
chunks = load_chunks(5)

embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)
vector_store = await build_vector_store(chunks, embedding_model, reset=False)
llm = build_llm("openai", model_name="gpt-4o-mini")

2026-03-05 15:30:45.038 | WARNING  | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:201 - Skipping unsupported file type '': .DS_Store


5


In [14]:
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=5,
    system_prompt=SYSTEM_PROMPT,
)

# Iteration

At each step, the current query, and the history is sent to the LLM for it to answer based on that. This can induce a big conversation, which leads to cost and potentially loose of performance. Here the previous sources are not sent, but that could be a (costly) option.

In [8]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent.answer(query_with_context)

print("Answer:")
print(f"{response.content}")
print(f"Sources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r}  |  {src.title!r}")

Answer:
[MessageContent(type='text', text='The following pallets in your portfolio have a third-party verified Environmental Product Declaration (EPD):\n\n1. **IPG** - Products 50-100, 50-101\n2. **CPR System** - Product 32-100\n3. **Relicyc** - Products 32-103\n4. **StabilPlastik** - Product 32-105\n5. **Redbox** - Product 11-100\n6. **Grupak** - Product 11-101\n\nThese suppliers have compliant EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).', image_url=None)]
Sources (5):
  'ART_internal_procurement_policy.pdf'  |  '### 3.2 Environmental Product Declarations (EPD)'
  'ART_internal_procurement_policy.pdf'  |  '## 2. Evidence Standards'
  'ART_internal_procurement_policy.pdf'  |  '### 3.4 Recycled Content Claims'
  'ART_internal_procurement_policy.pdf'  |  '### 3.1 All Suppliers and Products'
  'ART_internal_procurement_policy.pdf'  |  '### 3.3 PFAS (Per- and Polyfluoroalkyl Substances)'


In [15]:
query_l = [
    "Which pallets in our portfolio have a third-party verified EPD?",
    "Rewrite what you just said as a pirate and very concisely.",
    "Make minutes of our conversation so far, you are 'pAIrate' and I'm AIvan",
]

history: list[LLMMessage] = []

for query in query_l:
    response = await agent.answer(QueryWithContext(query=query, history=history))

    print("Answer:")
    print(f"{response.content}")
    print(f"\nSources ({len(response.sources)}):")
    for src in response.sources:
        source_file = src.metadata.get("source_file", "?")
        print(f"  {source_file!r}  |  {src.title!r}")

    response_content = response.content[0].text
    history.append(
        LLMMessage(content=[MessageContent(type="text", text=query)], role=Roles.USER)
    )
    history.append(
        LLMMessage(
            content=[MessageContent(type="text", text=response_content)],
            role=Roles.ASSISTANT,
        )
    )

    print("-" * 80)

Answer:
[MessageContent(type='text', text='The following pallets in your portfolio have a third-party verified Environmental Product Declaration (EPD):\n\n1. **IPG** - Products 50-100, 50-101\n2. **CPR System** - Product 32-100\n3. **Relicyc** - Product 32-103\n4. **StabilPlastik** - Product 32-105\n5. **Redbox** - Product 11-100\n6. **Grupak** - Product 11-101\n\nThese suppliers have compliant EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).', image_url=None)]

Sources (5):
  'ART_internal_procurement_policy.pdf'  |  '### 3.2 Environmental Product Declarations (EPD)'
  'ART_internal_procurement_policy.pdf'  |  '## 2. Evidence Standards'
  'ART_internal_procurement_policy.pdf'  |  '### 3.4 Recycled Content Claims'
  'ART_internal_procurement_policy.pdf'  |  '### 3.1 All Suppliers and Products'
  'ART_internal_procurement_policy.pdf'  |  '### 3.3 PFAS (Per- and Polyfluoroalkyl Substances)'
-------------------------------------------------------------------

In [16]:
history

[LLMMessage(content=[MessageContent(type='text', text='Which pallets in our portfolio have a third-party verified EPD?', image_url=None)], role=<Roles.USER: 'user'>, tool_calls=None, tool_call_id=None, name=None),
 LLMMessage(content=[MessageContent(type='text', text='The following pallets in your portfolio have a third-party verified Environmental Product Declaration (EPD):\n\n1. **IPG** - Products 50-100, 50-101\n2. **CPR System** - Product 32-100\n3. **Relicyc** - Product 32-103\n4. **StabilPlastik** - Product 32-105\n5. **Redbox** - Product 11-100\n6. **Grupak** - Product 11-101\n\nThese suppliers have compliant EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).', image_url=None)], role=<Roles.ASSISTANT: 'assistant'>, tool_calls=None, tool_call_id=None, name=None),
 LLMMessage(content=[MessageContent(type='text', text='Rewrite what you just said as a pirate and very concisely.', image_url=None)], role=<Roles.USER: 'user'>, tool_calls=None, tool_call_id=

# History summarization

One solution is to ask an LLM to summarize the conversation before each query and only send this summary to the LLM, potentially asking for a summary w.r.t. query. Depending on the implementation, it can reduce cost or improve performance. But one should be careful, as some relevant information might be lost.

In [17]:
async def summarize_history(history: list[LLMMessage]) -> str:
    llm_summarization = build_llm("openai", model_name="gpt-4o-mini")

    summarization_system_prompt = "You are a helpful assistant that summarizes conversations between a user and an assistant. Do not answer any question, just summarize the conversation in a concise way. The user is called 'User' and the assistant is called 'Assistant'."
    messages: list[LLMMessage] = [
        LLMMessage(
            content=[MessageContent(type="text", text=summarization_system_prompt)],
            role=Roles.SYSTEM,
        ),
        *history,
    ]

    msg = await llm_summarization.generate(conversation=messages)
    return msg.content

In [19]:
query_l = [
    "Which pallets in our portfolio have a third-party verified EPD?",
    "Rewrite what you just said as a pirate and very concisely.",
    "What is the current customer request?",
]

history: list[LLMMessage] = []
summaries: list[str] = []

for query in query_l:
    if len(history):
        summary_msg = await summarize_history(history)
        print(f"Summary: {summary_msg}")
        summaries.append(summary_msg)
        print("x" * 80)
        history_tmp = [LLMMessage(content=summary_msg, role=Roles.SYSTEM)]
    else:
        history_tmp = []

    response = await agent.answer(QueryWithContext(query=query, history=history_tmp))
    response_content = response.content[0].text

    print("Answer:")
    print(f"{response_content}")
    print(f"\nSources ({len(response.sources)}):")
    for src in response.sources:
        source_file = src.metadata.get("source_file", "?")
        print(f"  {source_file!r}  |  {src.title!r}")

    history.append(
        LLMMessage(content=[MessageContent(type="text", text=query)], role=Roles.USER)
    )
    history.append(
        LLMMessage(
            content=[MessageContent(type="text", text=response_content)],
            role=Roles.ASSISTANT,
        )
    )

    print("-" * 80)

Answer:
The following pallets in your portfolio have a third-party verified Environmental Product Declaration (EPD):

1. **IPG** - Products 50-100, 50-101
2. **CPR System** - Product 32-100
3. **Relicyc** - Product 32-103
4. **StabilPlastik** - Product 32-105
5. **Redbox** - Product 11-100
6. **Grupak** - Product 11-101

These suppliers have compliant EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).

Sources (5):
  'ART_internal_procurement_policy.pdf'  |  '### 3.2 Environmental Product Declarations (EPD)'
  'ART_internal_procurement_policy.pdf'  |  '## 2. Evidence Standards'
  'ART_internal_procurement_policy.pdf'  |  '### 3.4 Recycled Content Claims'
  'ART_internal_procurement_policy.pdf'  |  '### 3.1 All Suppliers and Products'
  'ART_internal_procurement_policy.pdf'  |  '### 3.3 PFAS (Per- and Polyfluoroalkyl Substances)'
--------------------------------------------------------------------------------
Summary: [MessageContent(type='text', text='User 

In [20]:
history

[LLMMessage(content=[MessageContent(type='text', text='Which pallets in our portfolio have a third-party verified EPD?', image_url=None)], role=<Roles.USER: 'user'>, tool_calls=None, tool_call_id=None, name=None),
 LLMMessage(content=[MessageContent(type='text', text='The following pallets in your portfolio have a third-party verified Environmental Product Declaration (EPD):\n\n1. **IPG** - Products 50-100, 50-101\n2. **CPR System** - Product 32-100\n3. **Relicyc** - Product 32-103\n4. **StabilPlastik** - Product 32-105\n5. **Redbox** - Product 11-100\n6. **Grupak** - Product 11-101\n\nThese suppliers have compliant EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).', image_url=None)], role=<Roles.ASSISTANT: 'assistant'>, tool_calls=None, tool_call_id=None, name=None),
 LLMMessage(content=[MessageContent(type='text', text='Rewrite what you just said as a pirate and very concisely.', image_url=None)], role=<Roles.USER: 'user'>, tool_calls=None, tool_call_id=

In [21]:
summaries

[[MessageContent(type='text', text='User inquired about which pallets in their portfolio have a third-party verified Environmental Product Declaration (EPD). Assistant provided a list of specific products from various suppliers that have compliant EPDs.', image_url=None)],
 [MessageContent(type='text', text='The user inquired about which pallets in their portfolio have a third-party verified Environmental Product Declaration (EPD). The assistant provided a list of specific pallets from various suppliers that meet this criterion, along with a source reference.', image_url=None)]]

# Query reformulation

The query from the user might need to be aware of the history to search correctly in the vector store. Query reformulation asks an LLM to rephrase the query to make it more relevant for search. This was actually already done under the hood for you. 

See conversational_toolkit.utils.retriever -> make_query_standalone

In [22]:
logger.add(sys.stderr, level="DEBUG")

5

In [24]:
query_l = [
    "Hi, I'm Ivan! What is the pallet which has a 3rd-party verified EPD?",
    "What is the meaning of code?",
]

history: list[LLMMessage] = []

for query in query_l:
    response = await agent.answer(QueryWithContext(query=query, history=history))
    response_content = response.content[0].text

    print("Answer:")
    print(f"{response_content}")
    print(f"\nSources ({len(response.sources)}):")
    for src in response.sources:
        source_file = src.metadata.get("source_file", "?")
        print(f"  {source_file!r}  |  {src.title!r}")

    history.append(
        LLMMessage(content=[MessageContent(type="text", text=query)], role=Roles.USER)
    )
    history.append(
        LLMMessage(
            content=[MessageContent(type="text", text=response_content)],
            role=Roles.ASSISTANT,
        )
    )

    print("-" * 80)

2026-03-05 15:36:03.581 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)


Answer:
The pallets that have a third-party verified Environmental Product Declaration (EPD) are:

1. Pallet 50-100 (Supplier: IPG)
2. Pallet 50-101 (Supplier: IPG)
3. Pallet 32-100 (Supplier: CPR System)
4. Pallet 32-103 (Supplier: Relicyc)
5. Pallet 32-105 (Supplier: StabilPlastik)
6. Pallet 11-100 (Supplier: Redbox)
7. Pallet 11-101 (Supplier: Grupak)

These pallets are compliant and have EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).

Sources (5):
  'ART_internal_procurement_policy.pdf'  |  '### 3.2 Environmental Product Declarations (EPD)'
  'ART_internal_procurement_policy.pdf'  |  '## 2. Evidence Standards'
  'ART_internal_procurement_policy.pdf'  |  '### 3.5 FSC and Other Chain-of-Custody Certifications'
  'ART_internal_procurement_policy.pdf'  |  '### 3.4 Recycled Content Claims'
  'ART_internal_procurement_policy.pdf'  |  '### 3.1 All Suppliers and Products'
--------------------------------------------------------------------------------


2026-03-05 15:36:07.726 | DEBUG    | conversational_toolkit.llms.openai:generate:114 - Completion: ChatCompletion(id='chatcmpl-DG463m7yJzLmePDkDjjgp3CoNvYKS', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='What is the meaning of Environmental Product Declaration (EPD)?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1772721367, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_373a14eb6f', usage=CompletionUsage(completion_tokens=12, prompt_tokens=389, total_tokens=401, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
2026-03-05 15:36:07.727 | INFO     | conversational_toolkit.llms.openai:generate:115 - LLM Usage: CompletionUsage(completion_tokens

Answer:
An Environmental Product Declaration (EPD) is a standardized document that provides transparent and comparable information about the environmental impact of a product throughout its life cycle. EPDs are based on a life cycle assessment (LCA) and must conform to specific standards, such as ISO 14025. They are verified by a third party to ensure that the claims made are credible and reliable. EPDs help consumers and businesses make informed decisions regarding the sustainability of products (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).

Sources (5):
  'ART_internal_procurement_policy.pdf'  |  '### 3.2 Environmental Product Declarations (EPD)'
  'ART_internal_procurement_policy.pdf'  |  '## 2. Evidence Standards'
  'ART_internal_procurement_policy.pdf'  |  '## 1. Purpose and Scope'
  'ART_internal_procurement_policy.pdf'  |  '### 3.1 All Suppliers and Products'
  'ART_internal_procurement_policy.pdf'  |  '### 3.3 PFAS (Per- and Polyfluoroalkyl Substances)'
----------------------

---------------------------